# A4 — Huấn luyện BRFSS 2015 theo Protocol V3

Notebook là giao diện điều khiển script A4, không sao chép thuật toán vào nhiều nơi. Chỉ chạy với CSV Kaggle gốc; dữ liệu `--quick` là smoke.

V3 implementation and command templates only. Primary SelectFromModel is implemented: weighted B3-configured XGBoost gain, exact k=21/15/10/5, stable original-column ties, per-fold fits and full-primary-training refits. Reserved validation selects k against locked full-feature A2 AP within 0.01; selected A3 is separately reported and A2 remains fallback. CV diagnostics are conditional on B3 settings selected on the same folds, not nested CV. ANOVA is separately saved QA only. Official training/evaluation authorization remains PENDING. F2 retains the ascending 199-point grid and smallest exact-tie threshold. All three paired AP comparisons use one locked V3 run. Historical B LightGBM scores are not official XGBoost evidence. SHAP defaults are 500/100; LIME remains 1000 perturbations.
Real **TRAINING-ONLY GroupSplit AP sensitivity** is complete: Stratified mean AP 0.4362740784; Grouped 0.4358097385; Group minus Stratified -0.0004643398; zero grouped-fold exact-X overlap. These descriptive training-CV results do not select the primary model or establish held-out performance. Official full-data A4/A5 and actual SHAP/LIME runtime remain pending. Verified numeric mapping is three-class 0/1/2 to binary 0/0/1; no labels changed.


## 1. Kiểm tra thư mục và file CSV

Use a reviewed **full repository copy**, including root `src/diabetes_research/`, component V3 config/amendment, scripts and both notebooks, `shared_splits/` archive/manifest/verifier, and root packaging/dependency metadata. Component-only mounting is insufficient. In a local clean copy, open from its root, component or notebooks directory. On Kaggle, copy the full repository to `/kaggle/working/UET.CS2046_7_Nhom5` (or another writable path), then set the notebook working directory to that root before the first code cell; discovery searches cwd and parents. Mount the original CSV separately and edit `DATA` in that cell to its mount path. Keep output paths writable and install editable from the repository root. These are reviewed templates; actual Kaggle reproduction and official A4/A5 remain pending.


In [ ]:
from pathlib import Path
# Discover a full reviewed repository copy from cwd and parents.
_required = (
    'src/diabetes_research/modeling.py',
    'src/diabetes_research/sensitivity_contract.py',
    'research/lead_modeling/a4_a5/config/A2_protocol_v3.yaml',
    'research/lead_modeling/a4_a5/config/PROTOCOL_AMENDMENTS_V3.json',
    'research/lead_modeling/a4_a5/scripts/run_a4.py',
    'research/lead_modeling/a4_a5/scripts/run_a5.py',
    'research/lead_modeling/a4_a5/scripts/run_group_sensitivity.py',
    'research/lead_modeling/a4_a5/notebooks/A4_Training_Protocol_v2.ipynb',
    'research/lead_modeling/a4_a5/notebooks/A5_Evaluation_XAI_v2.ipynb',
    'research/lead_modeling/a4_a5/requirements.txt',
    'shared_splits/shared_split_indices.npz',
    'shared_splits/shared_split_manifest.json',
    'shared_splits/verify_shared_split.py',
    'pyproject.toml',
    'requirements.txt',
)
_here = Path.cwd().resolve()
REPO = None
for _candidate in (_here, *_here.parents):
    if all((_candidate / _relative).is_file() for _relative in _required):
        REPO = _candidate
        break
if REPO is None:
    raise FileNotFoundError('Use a full V3 repository copy (root src, component config/scripts/notebooks, shared_splits and packaging); set cwd inside that copy before this cell')
ROOT = REPO / 'research' / 'lead_modeling' / 'a4_a5'
DATA = ROOT / 'data' / 'diabetes_binary_health_indicators_BRFSS2015.csv'
assert DATA.is_file(), f'Chưa có file {DATA}'
print('Dataset:', DATA)

## 2. Kiểm tra dataset

Xác minh 21 đặc trưng, tên target và tỷ lệ dương trước khi train.

In [ ]:
import sys
sys.path.insert(0, str(REPO / 'src'))
from diabetes_research.core import load_dataset, dataset_audit
X,y = load_dataset(DATA)
dataset_audit(X,y)
from diabetes_research.shared_splits import verify_shared_split
_, SPLIT_VERIFICATION = verify_shared_split(DATA)
SPLIT_VERIFICATION


## 3. Chạy experiment (nhiều CV fit, có thể tốn thời gian)

Đổi tên OUTPUT nếu muốn chạy lại. Mặc định 10 trial/họ mô hình, 3 folds.

In [ ]:
import subprocess, sys
OUTPUT = ROOT / 'results' / 'a4_v3_001'
cmd = [sys.executable, str(ROOT/'scripts'/'run_a4.py'),
       '--data',str(DATA),'--output-dir',str(OUTPUT), '--trials','10','--cv','3',
       '--shared-split-indices',str(REPO/'shared_splits/shared_split_indices.npz'),
       '--shared-split-manifest',str(REPO/'shared_splits/shared_split_manifest.json')]
subprocess.run(cmd, check=True)

## 4. Xem kết quả thực nghiệm

Không chỉnh hyperparameter hoặc ngưỡng bằng kết quả test.

In [ ]:
import pandas as pd
display(pd.read_csv(OUTPUT/'cv_results.csv').head())
display(pd.read_csv(OUTPUT/'feature_selection_cv.csv'))
display(pd.read_csv(OUTPUT/'validation_metrics.csv'))
display(pd.read_csv(OUTPUT/'test_metrics.csv'))


In [ ]:
import json
print(json.dumps(json.loads((OUTPUT/'selection.json').read_text()),indent=2,ensure_ascii=False))
print(json.dumps(json.loads((OUTPUT/'feature_rankings.json').read_text()),indent=2))
print(json.dumps(json.loads((OUTPUT/'secondary_anova.json').read_text()),indent=2))  # separate QA only
